# Pen2STEP - reviewer runs (rigor + significance)

Comprehensive real-set run for the Q1 revision. Runs all five conditions in one pass so every condition has **per-sample** data, then computes bootstrap 95% confidence intervals, per-family and dimensioned/undimensioned breakdowns, and paired significance tests (McNemar on validity, Wilcoxon on F-score).

- **`raw_mv`** : composed three-view sheet, frozen model, NO deterministic shell (real-data baseline).
- **`multiview` / `multiview_n3`** : wrapper with N=1 and N=3 (equal-N control).
- **`full_ocr` / `full_gt`** : dimension grounding via OCR and via ground truth.

**Before running:** Runtime > Change runtime type > **A100 GPU** (13B model in bf16 needs ~26 GB; A100-40GB fits, L4/T4 do not). Use A100 + bf16 to stay comparable with the existing numbers.

Note: Colab is only GPU compute to produce evaluation numbers. No training happens and the model stays frozen, so the training-free / offline claim is unaffected.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Dependencies. torch/numpy/Pillow are preinstalled on Colab; do NOT reinstall torch.
# svglib+reportlab are needed because src/render.py imports them at module load.
# The two conditions here do not use OCR, so easyocr is intentionally skipped.
!pip install -q "transformers>=4.39" "accelerate>=0.27" bitsandbytes "cadquery>=2.4.0" scipy svglib reportlab
print('deps installed')

In [ ]:
# Upload pen2step_eval.zip (built locally with make_colab_zip.py)
import os
from google.colab import files
up = files.upload()                       # pick pen2step_eval.zip
name = next(iter(up))
!rm -rf /content/proj && mkdir -p /content/proj && unzip -q "$name" -d /content/proj
os.chdir('/content/proj')
print('cwd:', os.getcwd())
!ls && echo '---' && ls data && echo '---' && ls data/real_eval

In [ ]:
# Comprehensive REAL run (n=19): all five conditions in one pass, so every
# condition has per-sample data for confidence intervals and paired tests.
# First run also downloads the model (~26 GB); roughly 20-25 min total on A100.
!python -m src.run_baselines --data data/real_eval --conditions raw_mv,multiview,multiview_n3,full_ocr,full_gt --limit 19

In [ ]:
# Statistical analysis on the real run: bootstrap 95% CIs, per-family and
# dimensioned/undimensioned breakdowns, and paired tests for the key comparisons.
!python -m src.analyze_results --json data/real_eval/results/ablation_n19.json --pairs raw_mv:multiview_n3,multiview_n3:full_ocr,multiview_n3:full_gt,full_ocr:full_gt

In [ ]:
# Show and download results + analysis (send these back).
import glob
print('=== REAL ablation ===')
for f in glob.glob('data/real_eval/results/*.md'):
    print(open(f, encoding='utf-8').read())
print('=== SYNTHETIC ===')
for f in glob.glob('data/results/*.md'):
    print(open(f, encoding='utf-8').read())
from google.colab import files
downloads = (glob.glob('data/real_eval/results/*.json')
             + glob.glob('data/real_eval/results/*_analysis.md')
             + glob.glob('data/results/*.json'))
for f in downloads:
    files.download(f)

In [ ]:
# Show and download results (send these back).
import glob
print('=== REAL ===')
for f in glob.glob('data/real_eval/results/*.md'):
    print(open(f, encoding='utf-8').read())
print('=== SYNTHETIC ===')
for f in glob.glob('data/results/*.md'):
    print(open(f, encoding='utf-8').read())
from google.colab import files
for f in glob.glob('data/real_eval/results/*.json') + glob.glob('data/results/*.json'):
    files.download(f)